In [1]:
import pandas as pd
import numpy as np


patients_path = '/media/Volume/data/MIMIC_IV/patients.csv.gz'
discharge_path = '/media/Volume/data/MIMIC_IV/discharge.csv.gz'
admissions_path = '/media/Volume/data/MIMIC_IV/admissions.csv.gz'
exams_path = '/media/Volume/data/MIMIC_IV/exams_filtered.csv'
out_path = '/media/Volume/data/MIMIC_IV/exams_cox_labels.csv'
icu_stays_path = '/media/Volume/data/MIMIC_IV/icustays.csv.gz'

code_15_exams = '/media/Volume/data/CODE15/processed/exams_filtered.csv'

In [2]:
code_15 = pd.read_csv(code_15_exams)
# drop unnamed 0 and unnamed 1
code_15 = code_15.loc[:, ~code_15.columns.str.contains('^Unnamed')]
# print columns
print(code_15.columns)
code_15.head(n=10)

Index(['exam_id', 'age', 'is_male', 'nn_predicted_age', '1dAVb', 'RBBB',
       'LBBB', 'SB', 'ST', 'AF', 'patient_id', 'death', 'timey', 'normal_ecg',
       'trace_file', 'r_peaks', 'r_peak_interval_mean', 'r_peak_variance',
       'valid'],
      dtype='object')


,exam_id,age,is_male,nn_predicted_age,1dAVb,RBBB,LBBB,SB,ST,AF,patient_id,death,timey,normal_ecg,trace_file,r_peaks,r_peak_interval_mean,r_peak_variance,valid
0,1169160,38,True,40.160484,False,False,False,False,False,False,523632,False,2.098628,True,exams_part13.hdf5,[ 57 285 507 742 972 1196 1419 1630 1848 ...,0.625309,0.020557,True
1,2873686,73,True,67.059440,False,False,False,False,False,False,1724173,False,6.657529,False,exams_part13.hdf5,[ 219 460 702 944 1176 1404 1631 1859 2080],0.646181,0.020969,True
2,168405,67,True,79.621740,False,False,False,False,False,True,51421,False,4.282188,False,exams_part13.hdf5,[ 220 532 716 1024 1200 1599 1874],0.765741,0.214907,True
3,271011,41,True,69.750260,False,False,False,False,False,False,1737282,False,4.038353,True,exams_part13.hdf5,[ 306 585 868 1142 1434 1725 2004 2290 2595 ...,0.795679,0.024149,True
4,384368,73,True,78.873460,False,False,False,False,False,False,331652,False,3.786298,False,exams_part13.hdf5,[ 115 285 454 623 793 962 1132 1303 1474 ...,0.473090,0.002729,True
5,2950575,61,True,70.905174,False,False,False,False,False,False,17423,NaN,NaN,False,exams_part13.hdf5,[ 248 530 817 1118 1409 1704 2005],0.813426,0.019406,True
6,1467619,33,False,48.628563,False,False,False,False,False,False,1351337,False,1.498629,False,exams_part13.hdf5,[ 271 623 969 1303 1629 1971 2313 2660],0.948016,0.022455,True
7,1537328,24,True,29.179337,False,False,False,False,False,False,1519774,False,1.367122,True,exams_part13.hdf5,[ 224 465 702 985 1259 1497 1744 2012 2269 ...,0.692222,0.054459,True
8,981735,82,True,82.224610,False,False,False,False,False,False,1192754,False,2.468491,False,exams_part13.hdf5,[ 160 432 684 925 1152 1377 1604 1838 2108 ...,0.716111,0.097054,True
9,132538,85,False,87.512890,False,False,False,False,False,True,60291,NaN,NaN,False,exams_part13.hdf5,[ 112 270 408 539 711 866 1053 1269 1505 ...,0.538095,0.123583,True


In [7]:
ecgs = pd.read_csv(exams_path)
print(ecgs.columns)

# keep columns we need
ecgs = ecgs[['subject_id', 'ecg_time', 'dod', 'anchor_year', 'anchor_age', 'fold', 'strat_fold']]
ecgs['dod'] = pd.to_datetime(ecgs['dod'], errors='coerce')
ecgs['ecg_time'] = pd.to_datetime(ecgs['ecg_time'], errors='coerce')
ecgs.head(n=5)

Index(['Unnamed: 0', 'file_name', 'study_id', 'subject_id', 'ecg_time',
       'ed_stay_id', 'ed_hadm_id', 'hosp_hadm_id', 'ed_diag_ed',
       'ed_diag_hosp', 'hosp_diag_hosp', 'all_diag_hosp', 'all_diag_all',
       'gender', 'age', 'anchor_year', 'anchor_age', 'dod',
       'ecg_no_within_stay', 'ecg_taken_in_ed', 'ecg_taken_in_hosp',
       'ecg_taken_in_ed_or_hosp', 'fold', 'strat_fold'],
      dtype='object')


,subject_id,ecg_time,dod,anchor_year,anchor_age,fold,strat_fold
0,10000032,2180-07-23 08:44:00,2180-09-09,2180.0,52.0,17,9
1,10000032,2180-07-23 09:54:00,2180-09-09,2180.0,52.0,17,9
2,10000032,2180-08-06 09:07:00,2180-09-09,2180.0,52.0,17,9
3,10000117,2181-03-04 17:14:00,NaT,2174.0,48.0,18,0
4,10000117,2183-09-18 13:52:00,NaT,2174.0,48.0,18,0


In [8]:
last_ecg_per_patient = ecgs.sort_values(by=['subject_id', 'ecg_time']).groupby('subject_id').last().reset_index()
last_ecg_per_patient = last_ecg_per_patient[['subject_id', 'ecg_time']]
last_ecg_per_patient.rename(columns={'ecg_time': 'last_ecg_time'}, inplace=True)
last_ecg_per_patient.head(n=5)

,subject_id,last_ecg_time
0,10000032,2180-08-06 09:07:00
1,10000117,2183-09-18 13:52:00
2,10000285,2159-11-26 14:29:00
3,10000560,2198-09-19 10:01:00
4,10000635,2139-01-22 14:09:00


In [26]:
# merge the two files on subject_id
merged = pd.merge(ecgs, last_ecg_per_patient, on='subject_id', how='left')

# keep only the columns we need
merged.head(n=10)

,subject_id,ecg_time,dod,anchor_year,anchor_age,fold,strat_fold,last_ecg_time
0,10000032,2180-07-23 08:44:00,2180-09-09,2180.0,52.0,17,9,2180-08-06 09:07:00
1,10000032,2180-07-23 09:54:00,2180-09-09,2180.0,52.0,17,9,2180-08-06 09:07:00
2,10000032,2180-08-06 09:07:00,2180-09-09,2180.0,52.0,17,9,2180-08-06 09:07:00
3,10000117,2181-03-04 17:14:00,NaT,2174.0,48.0,18,0,2183-09-18 13:52:00
4,10000117,2183-09-18 13:52:00,NaT,2174.0,48.0,18,0,2183-09-18 13:52:00
5,10000285,2159-11-26 14:29:00,NaT,2159.0,34.0,7,10,2159-11-26 14:29:00
6,10000560,2189-10-03 12:54:00,NaT,2189.0,53.0,7,4,2198-09-19 10:01:00
7,10000560,2198-09-19 10:01:00,NaT,2189.0,53.0,7,4,2198-09-19 10:01:00
8,10000635,2136-06-19 07:24:00,NaT,2136.0,74.0,7,19,2139-01-22 14:09:00
9,10000635,2136-06-20 08:54:00,NaT,2136.0,74.0,7,19,2139-01-22 14:09:00


In [27]:
merged['timey'] = merged['dod'].copy()
merged['death'] = merged['dod'].notna()
merged['timey'] = merged['timey'].fillna(merged['last_ecg_time'] + pd.Timedelta(days=365))
# convert timey to number of years
merged['timey'] = (merged['timey'] - merged['ecg_time']).dt.total_seconds() / (365.25 * 24 * 60 * 60)

merged.head(n=30)

,subject_id,ecg_time,dod,anchor_year,anchor_age,fold,strat_fold,last_ecg_time,timey,death
0,10000032,2180-07-23 08:44:00,2180-09-09,2180.0,52.0,17,9,2180-08-06 09:07:00,0.130421,True
1,10000032,2180-07-23 09:54:00,2180-09-09,2180.0,52.0,17,9,2180-08-06 09:07:00,0.130287,True
2,10000032,2180-08-06 09:07:00,2180-09-09,2180.0,52.0,17,9,2180-08-06 09:07:00,0.092047,True
3,10000117,2181-03-04 17:14:00,NaT,2174.0,48.0,18,0,2183-09-18 13:52:00,3.539657,False
4,10000117,2183-09-18 13:52:00,NaT,2174.0,48.0,18,0,2183-09-18 13:52:00,0.999316,False
5,10000285,2159-11-26 14:29:00,NaT,2159.0,34.0,7,10,2159-11-26 14:29:00,0.999316,False
6,10000560,2189-10-03 12:54:00,NaT,2189.0,53.0,7,4,2198-09-19 10:01:00,9.959972,False
7,10000560,2198-09-19 10:01:00,NaT,2189.0,53.0,7,4,2198-09-19 10:01:00,0.999316,False
8,10000635,2136-06-19 07:24:00,NaT,2136.0,74.0,7,19,2139-01-22 14:09:00,3.592830,False
9,10000635,2136-06-20 08:54:00,NaT,2136.0,74.0,7,19,2139-01-22 14:09:00,3.589921,False


In [ ]:
last_ecg = merged.sort_values(by=['subject_id', 'ecg_time']).groupby('subject_id').last().reset_index()
last_ecg = last_ecg[['subject_id', 'ecg_time', 'dod']]
# rename columns
last_ecg.rename(columns={'ecg_time': 'last_ecg_time'}, inplace=True)

In [ ]:
# we need another column where we have the time of death or end of follow-up
merged['died'] = np.where(merged['dod'].isna(), 0, 1)
# convert dod to datetime
merged['dod'] = pd.to_datetime(merged['dod'], errors='coerce')
merged['charttime'] = pd.to_datetime(merged['charttime'], errors='coerce')
merged['ecg_time'] = pd.to_datetime(merged['ecg_time'], errors='coerce')
merged['dod'] = merged['dod'].fillna(merged['charttime'] + pd.Timedelta(days=365))
# rename dod to follow_up_time
merged.rename(columns={'dod': 'follow_up_time'}, inplace=True)
merged.head(n=20)

,subject_id,ecg_time,follow_up_time,anchor_year,anchor_age,fold,strat_fold,note_id,charttime,died
0,10000032,2180-07-23 08:44:00,2180-09-09 00:00:00,2180.0,52.0,17,9,10000032-DS-24,2180-08-07,1
1,10000032,2180-07-23 09:54:00,2180-09-09 00:00:00,2180.0,52.0,17,9,10000032-DS-24,2180-08-07,1
2,10000032,2180-08-06 09:07:00,2180-09-09 00:00:00,2180.0,52.0,17,9,10000032-DS-24,2180-08-07,1
3,10000117,2181-03-04 17:14:00,2184-09-20 00:00:00,2174.0,48.0,18,0,10000117-DS-22,2183-09-21,1
4,10000117,2183-09-18 13:52:00,2184-09-20 00:00:00,2174.0,48.0,18,0,10000117-DS-22,2183-09-21,1
5,10000285,2159-11-26 14:29:00,2160-11-25 14:29:00,2159.0,34.0,7,10,NaN,NaT,0
6,10000560,2189-10-03 12:54:00,2190-10-17 00:00:00,2189.0,53.0,7,4,10000560-DS-15,2189-10-17,1
7,10000560,2198-09-19 10:01:00,2190-10-17 00:00:00,2189.0,53.0,7,4,10000560-DS-15,2189-10-17,1
8,10000635,2136-06-19 07:24:00,2137-06-19 07:24:00,2136.0,74.0,7,19,NaN,NaT,0
9,10000635,2136-06-20 08:54:00,2137-06-20 08:54:00,2136.0,74.0,7,19,NaN,NaT,0


In [ ]:
# if 